# RUL Sequence Models — LSTM and GRU

## Objective

This notebook evaluates sequence-based deep learning models for battery Remaining Useful Life (RUL) prediction using the frozen NASA battery feature dataset and official RUL targets.

The experiment uses a sequence length of 10 consecutive battery cycles and the 15 frozen predictor features defined in the ML Contract.

The primary validation strategy is Leave-One-Battery-Out (LOGO). Each test battery is completely excluded from model training and feature scaling for that fold.

### Models

- LSTM
- GRU

### Validation

- B0005 held out
- B0006 held out
- B0018 held out
- B0007 excluded from supervised RUL evaluation because it has no valid RUL target

### Metrics

- MAE
- RMSE
- R²

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\Renuka\Desktop\BatteryIQ


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import StandardScaler

from src.rul.sequence_dataset import (
    load_sequence_data,
    prepare_valid_rul_data,
    create_sequences,
)

from src.rul.lstm_model import (
    build_lstm_model,
    fit_scaler,
    transform_sequences,
    train_lstm_model,
    evaluate_lstm_model,
)

from src.rul.gru_model import (
    build_gru_model,
    train_gru_model,
    evaluate_gru_model,
)

In [3]:
# Load the frozen cycle-level feature and RUL datasets
df = load_sequence_data()

# Keep only cycles with valid official RUL targets
df_valid = prepare_valid_rul_data(df)

print("Full merged dataset shape:", df.shape)
print("Valid RUL dataset shape:", df_valid.shape)

print("\nBatteries included in supervised RUL data:")
print(df_valid["battery_id"].unique())

print("\nRows per battery:")
print(df_valid.groupby("battery_id").size())

Full merged dataset shape: (636, 19)
Valid RUL dataset shape: (331, 19)

Batteries included in supervised RUL data:
['B0005' 'B0006' 'B0018']

Rows per battery:
battery_id
B0005    125
B0006    109
B0018     97
dtype: int64


In [4]:
# Create fixed-length sequences without crossing battery boundaries
X, y, battery_ids = create_sequences(
    df_valid,
    sequence_length=10
)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Battery IDs shape:", battery_ids.shape)

print("\nSequences by battery:")
print(pd.Series(battery_ids).value_counts().sort_index())

X shape: (304, 10, 15)
y shape: (304,)
Battery IDs shape: (304,)

Sequences by battery:
B0005    116
B0006    100
B0018     88
Name: count, dtype: int64


In [5]:
# Batteries with valid supervised RUL targets
test_batteries = ["B0005", "B0006", "B0018"]

print("LOGO validation folds:")

for test_battery in test_batteries:
    train_batteries = [
        battery for battery in test_batteries
        if battery != test_battery
    ]

    print(
        f"Test: {test_battery} | "
        f"Train: {train_batteries}"
    )

LOGO validation folds:
Test: B0005 | Train: ['B0006', 'B0018']
Test: B0006 | Train: ['B0005', 'B0018']
Test: B0018 | Train: ['B0005', 'B0006']


In [6]:
# Run the complete LSTM Leave-One-Battery-Out experiment

logo_results = []

for test_battery in test_batteries:

    print(f"\n{'=' * 60}")
    print(f"Testing on: {test_battery}")
    print(f"{'=' * 60}")

    # Split by battery
    train_mask = np.isin(
        battery_ids,
        [battery for battery in test_batteries if battery != test_battery]
    )
    test_mask = battery_ids == test_battery

    X_train = X[train_mask]
    y_train = y[train_mask]

    X_test = X[test_mask]
    y_test = y[test_mask]

    # Fit scaler ONLY on training batteries
    scaler = fit_scaler(X_train)

    X_train_scaled = transform_sequences(X_train, scaler)
    X_test_scaled = transform_sequences(X_test, scaler)

    # Train a fresh LSTM for this fold
    model = train_lstm_model(
        X_train_scaled,
        y_train,
        epochs=50,
        batch_size=16,
    )

    # Evaluate on unseen test battery
    metrics, predictions = evaluate_lstm_model(
        model,
        X_test_scaled,
        y_test,
    )

    logo_results.append({
        "test_battery": test_battery,
        "train_batteries": [
            battery for battery in test_batteries
            if battery != test_battery
        ],
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics["R2"],
    })

    print(f"MAE :  {metrics['MAE']:.4f}")
    print(f"RMSE:  {metrics['RMSE']:.4f}")
    print(f"R²  :  {metrics['R2']:.4f}")

logo_results_df = pd.DataFrame(logo_results)

print("\n\nLSTM LOGO Results")
print(logo_results_df)


Testing on: B0005


MAE :  16.8644
RMSE:  19.3562
R²  :  0.6659

Testing on: B0006
MAE :  17.0740
RMSE:  20.6077
R²  :  0.4903

Testing on: B0018
MAE :  20.8928
RMSE:  25.3959
R²  :  0.0005


LSTM LOGO Results
  test_battery train_batteries        MAE       RMSE        R2
0        B0005  [B0006, B0018]  16.864388  19.356199  0.665853
1        B0006  [B0005, B0018]  17.074017  20.607706  0.490336
2        B0018  [B0005, B0006]  20.892836  25.395867  0.000465


In [7]:
gru_results = []

for test_battery in test_batteries:

    train_batteries = [
        battery
        for battery in test_batteries
        if battery != test_battery
    ]

    # Split sequences by battery
    train_mask = np.isin(
        battery_ids,
        train_batteries
    )

    test_mask = battery_ids == test_battery

    X_train = X[train_mask]
    y_train = y[train_mask]

    X_test = X[test_mask]
    y_test = y[test_mask]

    # Fit scaler ONLY on training data
    scaler = fit_scaler(X_train)

    X_train_scaled = transform_sequences(
        X_train,
        scaler
    )

    X_test_scaled = transform_sequences(
        X_test,
        scaler
    )

    # Train a fresh GRU for this fold
    model = train_gru_model(
        X_train_scaled,
        y_train,
        epochs=50,
        batch_size=16
    )

    # Evaluate on unseen battery
    metrics, predictions = evaluate_gru_model(
        model,
        X_test_scaled,
        y_test
    )

    gru_results.append({
        "Test Battery": test_battery,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R²": metrics["R2"]
    })


gru_results_df = pd.DataFrame(gru_results)

print("GRU LOGO Summary")
print("----------------")
print(
    gru_results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

GRU LOGO Summary
----------------
Test Battery       MAE      RMSE       R²
       B0005 22.737305 26.425161 0.377223
       B0006 16.090204 19.244781 0.555522
       B0018 18.665236 22.161268 0.238866


In [8]:
lstm_results_df = pd.DataFrame(logo_results)

lstm_results_df = lstm_results_df.rename(
    columns={
        "test_battery": "Test Battery",
        "R2": "R²"
    }
)

lstm_results_df["Model"] = "LSTM"

lstm_results_df = lstm_results_df[
    ["Model", "Test Battery", "MAE", "RMSE", "R²"]
]


gru_comparison_df = gru_results_df.copy()
gru_comparison_df["Model"] = "GRU"

gru_comparison_df = gru_comparison_df[
    ["Model", "Test Battery", "MAE", "RMSE", "R²"]
]


comparison_df = pd.concat(
    [
        lstm_results_df,
        gru_comparison_df
    ],
    ignore_index=True
)


print("LSTM vs GRU — LOGO Comparison")
print("--------------------------------")

print(
    comparison_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

LSTM vs GRU — LOGO Comparison
--------------------------------
Model Test Battery       MAE      RMSE       R²
 LSTM        B0005 16.864388 19.356199 0.665853
 LSTM        B0006 17.074017 20.607706 0.490336
 LSTM        B0018 20.892836 25.395867 0.000465
  GRU        B0005 22.737305 26.425161 0.377223
  GRU        B0006 16.090204 19.244781 0.555522
  GRU        B0018 18.665236 22.161268 0.238866


In [9]:
aggregate_comparison = (
    comparison_df
    .groupby("Model")[["MAE", "RMSE", "R²"]]
    .mean()
    .reset_index()
)

print("Aggregate LSTM vs GRU Results")
print("--------------------------------")

print(
    aggregate_comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

Aggregate LSTM vs GRU Results
--------------------------------
Model       MAE      RMSE       R²
  GRU 19.164248 22.610403 0.390537
 LSTM 18.277080 21.786591 0.385551
